# VAE-generated ensemble

## Experiment output folder

In [ ]:
from pathlib import Path

# Define the subfolder path for outputs
output_dir = Path("output/L16B06")

## Load a pre-trained VAE

In [ ]:
import torch
from modules.model import VariationalAutoencoder
from modules.dataset import MinMaxScale

## Load weight parameters and some metadata
fname = output_dir / 'model.pt'
checkpoint = torch.load(fname)

## Recreate the model
model = VariationalAutoencoder(checkpoint['LATENT_DIM'])
model.load_state_dict(checkpoint['model_state_dict'])

## Generate a VAE ensemble

In [ ]:
## Normalization
min_value = checkpoint['MINVAL']
max_value = checkpoint['MAXVAL']
transform = MinMaxScale(min_value, max_value)

## Generate nens new samples
nens = 12
z = torch.randn(nens, checkpoint['LATENT_DIM'])
with torch.no_grad():
    new_sample = model.decode(z)
    x = transform.invert(new_sample).squeeze()

## Plot configuration

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as crs                        # coordinate systems for maps
import cartopy.feature as cfeature

plt.rcParams['figure.dpi'] = 200

BORDERS = cfeature.NaturalEarthFeature(
        scale     = '10m',
        category  = 'cultural',
        name      = 'admin_0_countries',
        edgecolor = 'gray',
        facecolor = 'none'
        )
LAND = cfeature.NaturalEarthFeature(
        'physical', 'land', '10m',
        edgecolor = 'none',
        facecolor = 'lightgrey',
        alpha     = 0.8
        )

conf = {
    'levels': [5,10,15,20,25,30,35,40,45,50,55,60],
    'cmap': 'hot',
    'extend': 'max',
}

In [ ]:
## Coordinate variables
import numpy as np

## Required for map plotting
lats = np.linspace(22,32,101)
lons = np.linspace(-22,-10,121)

x = x.numpy()

## Plot ensemble mean

In [ ]:
nrows, ncols = 3, 4
fig, axs = plt.subplots(nrows   = nrows,
                        ncols   = ncols,
                        sharex  = True,
                        sharey  = True,
                        figsize = (12,7),
                        subplot_kw={'projection': crs.PlateCarree()}, 
                       )
fig.subplots_adjust(wspace=0.1, hspace=0.1)

labels = (c for c in 'abcdefghijklmm')
for i, ax in enumerate(axs.flat):
    cs = ax.contourf(lons,lats,x[i],**conf)
    ax.set_title(f'({next(labels)})', fontsize=10)

fig.colorbar(cs, 
             label = r'Ash column mass [$g/m^{2}$]', 
             shrink=0.4,
             ax=axs, 
            )

for i, ax in enumerate(axs.flat):
    ax.set_extent([-20, -11, 23, 30]) # [x1,x2,y1,y2]
    ax.add_feature(LAND,zorder=0)
    ax.add_feature(BORDERS, linewidth=0.4)
    ###
    ### Enables axis labels
    ###
    row, col = np.unravel_index(i, axs.shape)
    active_labels = []
    if col == 0:
        active_labels.append('left')
    if row == nrows - 1:
        active_labels.append('bottom')
    ###
    ### Add grid lines
    ###
    gl = ax.gridlines(
        crs         = crs.PlateCarree(),
        draw_labels = active_labels,
        linewidth   = 0.5,
        color       = 'gray',
        alpha       = 0.5,
        linestyle   = '--')
    gl.xlabel_style  = {'size': 8}
    gl.ylabel_style  = {'rotation': 89, 'size': 8}